# Text Summarization and Question Answering using T5 and BART

This notebook demonstrates advanced NLP tasks using transformer models:

## 🎯 **What You'll Learn:**
- **Text Summarization** with T5 and BART models
- **Question Answering** using T5
- **Extractive vs Abstractive** summarization techniques
- **Batch processing** for multiple texts
- **Interactive analysis** tools

## 🤖 **Models Covered:**
- **T5 (Text-To-Text Transfer Transformer)**: Versatile model treating all NLP tasks as text-to-text
- **BART (Bidirectional and Auto-Regressive Transformers)**: Excellent for text generation and summarization

## 📋 **Features:**
- ✅ Multiple model sizes (small, base, large)
- ✅ Customizable summary lengths
- ✅ Context-based question answering
- ✅ Extractive and abstractive summarization
- ✅ Batch processing capabilities
- ✅ Interactive testing tools

In [10]:
# Import Required Libraries
import torch
from transformers import (
    T5Tokenizer, T5ForConditionalGeneration,
    BartTokenizer, BartForConditionalGeneration,
    pipeline
)
from typing import List, Dict, Union, Optional
import re
import warnings
warnings.filterwarnings('ignore')

# Check package versions and GPU availability
print("📦 Package Information:")
print(f"PyTorch version: {torch.__version__}")
print(f"Device available: {'GPU' if torch.cuda.is_available() else 'CPU'}")

try:
    import transformers
    print(f"Transformers version: {transformers.__version__}")
    print("✅ All required packages are available!")
except ImportError as e:
    print(f"❌ Missing package: {e}")
    print("Install with: pip install transformers torch datasets")


import requests
from huggingface_hub import configure_http_backend

def backend_factory() -> requests.Session:
    session = requests.Session()
    session.verify = False
    return session

configure_http_backend(backend_factory=backend_factory)

📦 Package Information:
PyTorch version: 2.9.0+cpu
Device available: CPU
Transformers version: 4.57.1
✅ All required packages are available!


## 📝 TextSummarizer Class

Let's build our text summarization system step by step. This class supports both T5 and BART models with different sizes.

In [11]:
# TextSummarizer Class - Initialization
class TextSummarizer:
    """
    A text summarization class supporting both T5 and BART models.
    
    Supported models:
    - T5: t5-small, t5-base, t5-large
    - BART: facebook/bart-large-cnn, facebook/bart-base-cnn
    """
    
    def __init__(self, model_type: str = "t5", model_size: str = "small"):
        """
        Initialize the text summarizer.
        
        Args:
            model_type (str): Either "t5" or "bart"
            model_size (str): Model size ("small", "base", "large")
        """
        self.model_type = model_type.lower()
        self.model_size = model_size
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        print(f"🔧 Initializing TextSummarizer")
        print(f"📱 Using device: {self.device}")
        print(f"🤖 Model type: {self.model_type}")
        print(f"📏 Model size: {self.model_size}")
        
        self.tokenizer = None
        self.model = None
        self.summarizer = None
        self.is_loaded = False

print("✅ TextSummarizer class defined!")

✅ TextSummarizer class defined!


In [12]:
# TextSummarizer - Model Loading Method
def load_model(self):
    """Load the appropriate model and tokenizer."""
    try:
        if self.model_type == "t5":
            model_name = f"t5-{self.model_size}"
            print(f"📥 Loading T5 model: {model_name}")
            
            self.tokenizer = T5Tokenizer.from_pretrained(model_name)
            self.model = T5ForConditionalGeneration.from_pretrained(model_name)
            
        elif self.model_type == "bart":
            if self.model_size == "small":
                model_name = "facebook/bart-large-cnn"  # No small version available
                print("⚠️ BART small not available, using large-cnn")
            else:
                model_name = f"facebook/bart-{self.model_size}-cnn"
            
            print(f"📥 Loading BART model: {model_name}")
            
            self.tokenizer = BartTokenizer.from_pretrained(model_name)
            self.model = BartForConditionalGeneration.from_pretrained(model_name)
        
        else:
            raise ValueError("model_type must be either 't5' or 'bart'")
        
        self.model.to(self.device)
        
        # Create pipeline for easier inference
        self.summarizer = pipeline(
            "summarization",
            model=self.model,
            tokenizer=self.tokenizer,
            device=0 if self.device.type == "cuda" else -1
        )
        
        self.is_loaded = True
        print("✅ Model loaded successfully!")
        print(f"📊 Model parameters: ~{sum(p.numel() for p in self.model.parameters()) / 1e6:.1f}M")
        
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        print("💡 Try using a smaller model or check your internet connection")
        self.is_loaded = False

# Add method to class
TextSummarizer.load_model = load_model

print("✅ Model loading method added!")

✅ Model loading method added!


In [13]:
# TextSummarizer - Single Text Summarization
def summarize(self, text: str, max_length: int = 150, min_length: int = 30) -> Dict[str, Union[str, int]]:
    """
    Summarize a given text.
    
    Args:
        text (str): Text to summarize
        max_length (int): Maximum length of summary
        min_length (int): Minimum length of summary
        
    Returns:
        dict: Summary result with metadata
    """
    if not self.is_loaded:
        return {"summary": "Error: Model not loaded", "original_length": 0, "summary_length": 0}
    
    if not text.strip():
        return {"summary": "", "original_length": 0, "summary_length": 0}
    
    try:
        # For T5, we need to add the task prefix
        if self.model_type == "t5":
            input_text = f"summarize: {text}"
        else:
            input_text = text
        
        # Generate summary
        summary_result = self.summarizer(
            input_text,
            max_length=max_length,
            min_length=min_length,
            do_sample=False,
            truncation=True
        )
        
        summary = summary_result[0]['summary_text']
        original_words = len(text.split())
        summary_words = len(summary.split())
        
        return {
            "summary": summary,
            "original_length": original_words,
            "summary_length": summary_words,
            "compression_ratio": round(summary_words / original_words, 2) if original_words > 0 else 0,
            "model_used": f"{self.model_type}-{self.model_size}"
        }
        
    except Exception as e:
        print(f"❌ Error during summarization: {e}")
        return {"summary": f"Error: {str(e)}", "original_length": 0, "summary_length": 0}

# Add method to class
TextSummarizer.summarize = summarize

print("✅ Single text summarization method added!")

✅ Single text summarization method added!


In [15]:
# TextSummarizer - Batch Processing
def summarize_batch(self, texts: List[str], max_length: int = 150, min_length: int = 30) -> List[Dict]:
    """
    Summarize multiple texts efficiently.
    
    Args:
        texts (List[str]): List of texts to summarize
        max_length (int): Maximum length of each summary
        min_length (int): Minimum length of each summary
        
    Returns:
        List[dict]: List of summary results
    """
    if not self.is_loaded:
        return [{"summary": "Error: Model not loaded"} for _ in texts]
    
    results = []
    print(f"📊 Processing {len(texts)} texts...")
    
    for i, text in enumerate(texts):
        print(f"⏳ Processing text {i+1}/{len(texts)}")
        result = self.summarize(text, max_length, min_length)
        result["text_index"] = i
        result["original_text"] = text[:100] + "..." if len(text) > 100 else text
        results.append(result)
    
    return results

# TextSummarizer - Extractive Summary
def extractive_summary(self, text: str, num_sentences: int = 3) -> str:
    """
    Create an extractive summary by selecting key sentences.
    
    Args:
        text (str): Input text
        num_sentences (int): Number of sentences to include
        
    Returns:
        str: Extractive summary
    """
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
    
    if len(sentences) <= num_sentences:
        return text
    
    # Simple scoring based on sentence length and position
    scores = []
    for i, sentence in enumerate(sentences):
        # Score based on length and position (earlier sentences get bonus)
        length_score = len(sentence.split())
        position_score = max(0, len(sentences) - i) / len(sentences)
        total_score = length_score + (position_score * 10)
        scores.append((total_score, i, sentence))
    
    # Sort by score and take top sentences
    top_sentences = sorted(scores, reverse=True)[:num_sentences]
    
    # Sort by original position to maintain order
    top_sentences = sorted(top_sentences, key=lambda x: x[1])
    
    return '. '.join([sentence[2] for sentence in top_sentences]) + '.'

# Add methods to class
TextSummarizer.summarize_batch = summarize_batch
TextSummarizer.extractive_summary = extractive_summary

print("✅ Batch processing and extractive summary methods added!")

✅ Batch processing and extractive summary methods added!


## 🚀 Initialize Models

Let's create instances of our summarizer and question answerer. You can modify the model types and sizes here.

In [16]:
# Initialize Text Summarizer
print("🔧 Creating Text Summarizer...")

# You can change these parameters:
# model_type: "t5" or "bart"
# model_size: "small", "base", "large"
summarizer = TextSummarizer(model_type="t5", model_size="small")

# Load the summarization model
print("\n📥 Loading summarization model...")
summarizer.load_model()

if summarizer.is_loaded:
    print(f"🎉 Text Summarizer ready!")
    print(f"🤖 Using: {summarizer.model_type}-{summarizer.model_size}")
else:
    print("⚠️ Summarizer failed to load")

🔧 Creating Text Summarizer...
🔧 Initializing TextSummarizer
📱 Using device: cpu
🤖 Model type: t5
📏 Model size: small

📥 Loading summarization model...
📥 Loading T5 model: t5-small


Device set to use cpu


✅ Model loaded successfully!
📊 Model parameters: ~60.5M
🎉 Text Summarizer ready!
🤖 Using: t5-small


## 📰 Test Text Summarization

Let's test our summarization system with sample text. We'll try both abstractive (AI-generated) and extractive (sentence selection) approaches.

In [20]:
# Sample Text for Summarization
sample_article = """
Artificial Intelligence (AI) has revolutionized numerous industries and aspects of human life. 
From healthcare to transportation, education to entertainment, AI technologies are transforming 
how we work, learn, and interact with the world around us. Machine learning, a subset of AI, 
enables systems to automatically learn and improve from experience without being explicitly programmed. 
Deep learning, which uses neural networks with multiple layers, has achieved remarkable breakthroughs 
in image recognition, natural language processing, and speech recognition.

The applications of AI are vast and growing. In healthcare, AI assists in medical diagnosis, 
drug discovery, and personalized treatment plans. Autonomous vehicles use AI for navigation, 
obstacle detection, and decision-making. In finance, AI helps with fraud detection, algorithmic 
trading, and risk assessment. Virtual assistants like Siri, Alexa, and Google Assistant use 
natural language processing to understand and respond to human queries.

However, the rapid advancement of AI also raises important ethical and societal concerns. 
Issues such as job displacement, privacy, bias in algorithms, and the potential for misuse 
of AI technologies need careful consideration. As AI becomes more prevalent, it's crucial 
to develop responsible AI practices and ensure that these technologies benefit humanity as a whole.
"""

print("📄 Original Article:")
print("=" * 50)
print(sample_article)
print(f"\n📊 Original length: {len(sample_article.split())} words")

result= summarizer.summarize(sample_article)

print("summary is",result)

Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📄 Original Article:

Artificial Intelligence (AI) has revolutionized numerous industries and aspects of human life. 
From healthcare to transportation, education to entertainment, AI technologies are transforming 
how we work, learn, and interact with the world around us. Machine learning, a subset of AI, 
enables systems to automatically learn and improve from experience without being explicitly programmed. 
Deep learning, which uses neural networks with multiple layers, has achieved remarkable breakthroughs 
in image recognition, natural language processing, and speech recognition.

The applications of AI are vast and growing. In healthcare, AI assists in medical diagnosis, 
drug discovery, and personalized treatment plans. Autonomous vehicles use AI for navigation, 
obstacle detection, and decision-making. In finance, AI helps with fraud detection, algorithmic 
trading, and risk assessment. Virtual assistants like Siri, Alexa, and Google Assistant use 
natural language processing to

## ❓ QuestionAnswerer Class

Now let's build the question answering system using T5, which excels at this task due to its text-to-text approach.

In [23]:
# QuestionAnswerer Class Definition
class QuestionAnswerer:
    """
    A question answering class using T5 model.
    """
    
    def __init__(self, model_size: str = "small"):
        """
        Initialize the question answerer.
        
        Args:
            model_size (str): Model size ("small", "base", "large")
        """
        self.model_size = model_size
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        print(f"🔧 Initializing QuestionAnswerer")
        print(f"📱 Using device: {self.device}")
        print(f"📏 Model size: {self.model_size}")
        
        self.tokenizer = None
        self.model = None
        self.is_loaded = False

def load_qa_model(self):
    """Load the T5 model for question answering."""
    try:
        model_name = f"t5-{self.model_size}"
        print(f"📥 Loading T5 model for QA: {model_name}")
        
        self.tokenizer = T5Tokenizer.from_pretrained(model_name)
        self.model = T5ForConditionalGeneration.from_pretrained(model_name)
        self.model.to(self.device)
        
        self.is_loaded = True
        print("✅ QA Model loaded successfully!")
        print(f"📊 Model parameters: ~{sum(p.numel() for p in self.model.parameters()) / 1e6:.1f}M")
        
    except Exception as e:
        print(f"❌ Error loading QA model: {e}")
        print("💡 Try using a smaller model or check your internet connection")
        self.is_loaded = False

# Add method to class
QuestionAnswerer.load_model = load_qa_model

print("✅ QuestionAnswerer class defined!")

✅ QuestionAnswerer class defined!


In [29]:
# QuestionAnswerer - Answer Single Question
def answer_question(self, context: str, question: str, max_length: int = 100) -> Dict[str, str]:
    """
    Answer a question based on given context.
    
    Args:
        context (str): Context text containing the answer
        question (str): Question to answer
        max_length (int): Maximum length of answer
        
    Returns:
        dict: Answer result with metadata
    """
    if not self.is_loaded:
        return {"question": question, "answer": "Error: Model not loaded", "context_length": 0}
    
    try:
        # Format input for T5
        input_text = f"question: {question} context: {context}"
        
        # Tokenize input
        inputs = self.tokenizer.encode(
            input_text,
            return_tensors="pt",
            max_length=512,
            truncation=True
        )
        inputs = inputs.to(self.device)
        
        # Generate answer
        with torch.no_grad():
            outputs = self.model.generate(
                inputs,
                max_length=max_length,
                num_return_sequences=1,
                temperature=0.7,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id
            )
        
        # Decode answer
        answer = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        return {
            "question": question,
            "answer": answer,
            "context_length": len(context.split()),
            "answer_length": len(answer.split()),
            "context_preview": context[:100] + "..." if len(context) > 100 else context
        }
        
    except Exception as e:
        print(f"❌ Error answering question: {e}")
        return {
            "question": question,
            "answer": f"Error: {str(e)}",
            "context_length": 0,
            "answer_length": 0
        }

def answer_multiple_questions(self, context: str, questions: List[str]) -> List[Dict[str, str]]:
    """
    Answer multiple questions based on the same context.
    
    Args:
        context (str): Context text
        questions (List[str]): List of questions
        
    Returns:
        List[dict]: List of answer results
    """
    if not self.is_loaded:
        return [{"question": q, "answer": "Error: Model not loaded"} for q in questions]
    
    results = []
    print(f"❓ Answering {len(questions)} questions...")
    
    for i, question in enumerate(questions):
        print(f"⏳ Processing question {i+1}/{len(questions)}")
        result = self.answer_question(context, question)
        results.append(result)
    
    return results

# Add methods to class
QuestionAnswerer.answer_question = answer_question
QuestionAnswerer.answer_multiple_questions = answer_multiple_questions

print("✅ Question answering methods added!")

✅ Question answering methods added!


In [30]:
# Initialize Question Answerer
print("🔧 Creating Question Answerer...")

# You can change the model size: "small", "base", "large"
qa_system = QuestionAnswerer(model_size="small")

# Load the QA model
print("\n📥 Loading QA model...")
qa_system.load_model()

if qa_system.is_loaded:
    print(f"🎉 Question Answerer ready!")
    print(f"🤖 Using: t5-{qa_system.model_size}")
else:
    print("⚠️ QA system failed to load")

🔧 Creating Question Answerer...
🔧 Initializing QuestionAnswerer
📱 Using device: cpu
📏 Model size: small

📥 Loading QA model...
📥 Loading T5 model for QA: t5-small
✅ QA Model loaded successfully!
📊 Model parameters: ~60.5M
🎉 Question Answerer ready!
🤖 Using: t5-small
✅ QA Model loaded successfully!
📊 Model parameters: ~60.5M
🎉 Question Answerer ready!
🤖 Using: t5-small


In [31]:
# Test QA System with Single Questions
print("🧪 Testing Question Answering System")
print("=" * 60)

# Check if QA system is loaded
if not qa_system.is_loaded:
    print("❌ QA System not loaded! Please run the QA initialization cell first.")
else:
    print(f"✅ QA System ready: t5-{qa_system.model_size}")

# Use the same AI article as context
context = """
Artificial Intelligence (AI) has revolutionized numerous industries and aspects of human life. 
From healthcare to transportation, education to entertainment, AI technologies are transforming 
how we work, learn, and interact with the world around us. Machine learning, a subset of AI, 
enables systems to automatically learn and improve from experience without being explicitly programmed. 
Deep learning, which uses neural networks with multiple layers, has achieved remarkable breakthroughs 
in image recognition, natural language processing, and speech recognition.

The applications of AI are vast and growing. In healthcare, AI assists in medical diagnosis, 
drug discovery, and personalized treatment plans. Autonomous vehicles use AI for navigation, 
obstacle detection, and decision-making. In finance, AI helps with fraud detection, algorithmic 
trading, and risk assessment. Virtual assistants like Siri, Alexa, and Google Assistant use 
natural language processing to understand and respond to human queries.

However, the rapid advancement of AI also raises important ethical and societal concerns. 
Issues such as job displacement, privacy, bias in algorithms, and the potential for misuse 
of AI technologies need careful consideration. As AI becomes more prevalent, it's crucial 
to develop responsible AI practices and ensure that these technologies benefit humanity as a whole.
"""

# Test individual questions
test_questions = [
    "What is machine learning?",
    "What are some applications of AI in healthcare?",
    "What ethical concerns does AI raise?",
    "What virtual assistants are mentioned in the text?",
    "How does deep learning work?"
]

print(f"\n📝 Context: AI Article ({len(context.split())} words)")
print("\n" + "="*60)

for i, question in enumerate(test_questions, 1):
    print(f"\n🔍 Question {i}: {question}")
    print("-" * 40)
    
    if qa_system.is_loaded:
        result = qa_system.answer_question(context, question)
        print(f"💡 Answer: {result['answer']}")
        print(f"📊 Answer length: {result['answer_length']} words")
    else:
        print("❌ Cannot answer - QA system not loaded")

print(f"\n✅ Single question testing completed!")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


🧪 Testing Question Answering System
✅ QA System ready: t5-small

📝 Context: AI Article (192 words)


🔍 Question 1: What is machine learning?
----------------------------------------
💡 Answer: a subset of AI
📊 Answer length: 4 words

🔍 Question 2: What are some applications of AI in healthcare?
----------------------------------------
💡 Answer: a subset of AI
📊 Answer length: 4 words

🔍 Question 2: What are some applications of AI in healthcare?
----------------------------------------
💡 Answer: medical diagnosis, drug discovery, and personalized treatment plans
📊 Answer length: 8 words

🔍 Question 3: What ethical concerns does AI raise?
----------------------------------------
💡 Answer: medical diagnosis, drug discovery, and personalized treatment plans
📊 Answer length: 8 words

🔍 Question 3: What ethical concerns does AI raise?
----------------------------------------
💡 Answer: job displacement, privacy, bias in algorithms, and the potential for misuse of AI technologies
📊 Answer leng